# Notebook 02 — BLIP Architecture Probe, Caption Generation & Teacher-Forced Token Representations

**Purpose:** establish the exact BLIP implementation used by the Major and produce the standardized prediction artifact required by later training and evaluation.

### Major conditions

1. **frozen_blip** — pretrained BLIP, no adaptation.
2. **lora_baseline** — BLIP with LoRA, caption loss only.
3. **consistency** — BLIP with LoRA, caption loss + targeted object-level consistency loss.

This notebook does **not** train the consistency objective. It verifies the architecture, generates the frozen baseline, and verifies that a teacher-forced BLIP forward pass exposes the contextual decoder states and cross-attention needed by the Major training pipeline.

### Critical distinction from the Minor

The Minor used generated-caption cross-attention for attention analysis. The Major semantic representation is the **contextual decoder hidden state under teacher forcing**. Cross-attention is retained here as a diagnostic/architecture probe, not as the semantic representation used by the consistency objective.


## 0. Architecture facts must be measured, not guessed

This notebook records the actual runtime values for:

- checkpoint and model class;
- decoder hidden size;
- teacher-forced last hidden-state shape;
- number of visual tokens;
- cross-attention shape when available;
- token sequence alignment;
- candidate LoRA target modules;
- whether one forward pass exposes hidden states and cross-attention together.

The Minor's `24 × 24` attention map was an analysis resize. It must not be interpreted as proof that BLIP natively has 576 visual tokens.


In [ ]:
# CELL 1 — Environment and imports
# Install requirements once in the repository environment. Do not pip-install in this notebook.

from pathlib import Path
import ast
import json
import random
import sys
import warnings

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import BlipProcessor, BlipForConditionalGeneration

warnings.filterwarnings("ignore")
PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import config as cfg

SEED = int(cfg.SEED)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(getattr(cfg, "DEVICE", "cuda" if torch.cuda.is_available() else "cpu"))
print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)
print("Torch:", torch.__version__)


In [ ]:
# CELL 2 — Resolve Major configuration
BLIP_CHECKPOINT = getattr(cfg, "BLIP_CHECKPOINT", "Salesforce/blip-image-captioning-large")
GENERATION_CONFIG = {
    "max_new_tokens": int(getattr(cfg, "MAX_NEW_TOKENS", 80)),
    "num_beams": int(getattr(cfg, "NUM_BEAMS", 5)),
    "length_penalty": float(getattr(cfg, "LENGTH_PENALTY", 1.1)),
    "no_repeat_ngram_size": int(getattr(cfg, "NO_REPEAT_NGRAM_SIZE", 2)),
    "early_stopping": bool(getattr(cfg, "EARLY_STOPPING", True)),
}
OUTPUT_DIR = Path(getattr(cfg, "OUTPUT_DIR", PROJECT_ROOT / "outputs"))
DIAGNOSTICS_DIR = Path(getattr(cfg, "DIAGNOSTICS_DIR", OUTPUT_DIR / "diagnostics"))
PREDICTIONS_DIR = Path(getattr(cfg, "PREDICTIONS_DIR", OUTPUT_DIR / "predictions"))
for p in [OUTPUT_DIR, DIAGNOSTICS_DIR, PREDICTIONS_DIR]: p.mkdir(parents=True, exist_ok=True)
print("BLIP checkpoint:", BLIP_CHECKPOINT)
print("Generation config:", GENERATION_CONFIG)


## 1. Load the exact BLIP checkpoint

The checkpoint must be the same checkpoint used by the Major training code. If it changes, rerun this notebook and regenerate the architecture report.


In [ ]:
# CELL 3 — Load processor and model
processor = BlipProcessor.from_pretrained(BLIP_CHECKPOINT)
model = BlipForConditionalGeneration.from_pretrained(BLIP_CHECKPOINT)
model.to(DEVICE).eval()
print("Model class:", type(model).__name__)
print("Processor class:", type(processor).__name__)
print("Text decoder config:", type(model.config.text_config).__name__)
print("Vision config:", type(model.config.vision_config).__name__)


In [ ]:
# CELL 4 — Print architecture dimensions without hard-coding them
def first_attr(obj, names):
    for name in names:
        if hasattr(obj, name) and getattr(obj, name) is not None:
            return getattr(obj, name)
    return None
vision_cfg = model.config.vision_config
text_cfg = model.config.text_config
hidden_size = first_attr(text_cfg, ["hidden_size", "d_model"])
vision_hidden_size = first_attr(vision_cfg, ["hidden_size", "vision_width"])
image_size = first_attr(vision_cfg, ["image_size"])
patch_size = first_attr(vision_cfg, ["patch_size"])
print("Decoder hidden size:", hidden_size)
print("Vision hidden size:", vision_hidden_size)
print("Vision image size:", image_size)
print("Vision patch size:", patch_size)
if image_size and patch_size:
    gh, gw = image_size // patch_size, image_size // patch_size
    print("Theoretical patch grid:", (gh, gw))
    print("Theoretical patch tokens:", gh * gw)
    print("Special token count: verify from runtime output")


## 2. Dataset manifest

Notebook 01 should produce a manifest containing at least `image_id` and `image_path`. No CDN fallback, random-noise image, or simulated image is allowed.


In [ ]:
# CELL 5 — Locate the manifest
candidates = [
    Path(getattr(cfg, "DATASET_MANIFEST", "")),
    OUTPUT_DIR / "dataset_manifest.csv",
    DIAGNOSTICS_DIR / "dataset_manifest.csv",
    PROJECT_ROOT / "data" / "dataset_manifest.csv",
]
candidates = [p for p in candidates if str(p) not in {"", "."}]
MANIFEST_PATH = next((p for p in candidates if p.exists()), None)
if MANIFEST_PATH is None:
    raise FileNotFoundError("No dataset manifest found. Run Notebook 01 and set DATASET_MANIFEST if needed.")
print("Using manifest:", MANIFEST_PATH)


In [ ]:
# CELL 6 — Load and validate manifest
df = pd.read_csv(MANIFEST_PATH)
required = {"image_id", "image_path"}
missing = required - set(df.columns)
if missing: raise ValueError(f"Manifest missing columns: {sorted(missing)}")
df["image_path"] = df["image_path"].astype(str)
missing_images = [p for p in df["image_path"] if not Path(p).exists()]
if missing_images:
    raise FileNotFoundError(f"{len(missing_images)} image paths do not exist. No fallback is permitted.")
print("Rows:", len(df), "Unique images:", df["image_id"].nunique())
display(df.head(3))


## 3. Runtime architecture probe with teacher forcing

Run a real image and a reference caption through BLIP with labels, hidden states, and attentions enabled. The runtime output determines the actual tensor structure.


In [ ]:
# CELL 7 — Select a real probe example
row0 = df.iloc[0]
probe_image = Image.open(row0["image_path"]).convert("RGB")

def get_reference_captions(row):
    for col in ["reference_captions", "ground_truth_caps"]:
        if col in row.index and pd.notna(row[col]):
            raw = row[col]
            if isinstance(raw, list): return [str(x) for x in raw]
            try:
                parsed = ast.literal_eval(str(raw))
                if isinstance(parsed, list): return [str(x) for x in parsed]
            except Exception: pass
            return [str(raw)]
    if "primary_gt_caption" in row.index and pd.notna(row["primary_gt_caption"]):
        return [str(row["primary_gt_caption"])]
    return []

probe_refs = get_reference_captions(row0)
if not probe_refs: raise ValueError("Probe image has no reference caption.")
probe_caption = probe_refs[0]
print("Probe caption:", probe_caption)


In [ ]:
# CELL 8 — Teacher-forced forward pass
inputs = processor(images=probe_image, text=probe_caption, return_tensors="pt", padding=True)
inputs = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k,v in inputs.items()}
labels = inputs["input_ids"].clone()
with torch.no_grad():
    outputs = model(
        pixel_values=inputs["pixel_values"],
        input_ids=inputs["input_ids"],
        attention_mask=inputs.get("attention_mask"),
        labels=labels,
        output_hidden_states=True,
        output_attentions=True,
        return_dict=True,
    )
print("Output type:", type(outputs).__name__)
for field in outputs.keys():
    value = getattr(outputs, field)
    if torch.is_tensor(value): print(field, tuple(value.shape))
    elif isinstance(value, (tuple,list)):
        print(field, "sequence length", len(value), "first shapes", [tuple(x.shape) if torch.is_tensor(x) else type(x).__name__ for x in value[:3]])
    elif value is not None: print(field, type(value).__name__)


In [ ]:
# CELL 9 — Resolve contextual decoder hidden states
last_hidden = getattr(outputs, "last_hidden_state", None)
if last_hidden is None:
    hs = getattr(outputs, "hidden_states", None)
    if hs is not None: last_hidden = hs[-1]
if last_hidden is None or last_hidden.ndim != 3:
    raise RuntimeError("Could not resolve decoder contextual states as [batch, sequence, hidden].")
B,T,D = last_hidden.shape
print("Contextual decoder state:", tuple(last_hidden.shape))
print("Runtime hidden dimension:", D)
if hidden_size is not None and int(hidden_size) != D:
    raise RuntimeError(f"Config hidden size {hidden_size} != runtime hidden size {D}")


## 4. Token-position alignment

The decoder hidden state position must not be assumed to be the same as a target-label position. The model uses shifted causal decoding internally. The next training notebook must explicitly account for this alignment when extracting token representations for eligible target words.


In [ ]:
# CELL 10 — Inspect token sequence
input_ids = inputs["input_ids"][0].detach().cpu().tolist()
mask = inputs.get("attention_mask")
mask = mask[0].detach().cpu().tolist() if mask is not None else [1]*len(input_ids)
tokens = processor.tokenizer.convert_ids_to_tokens(input_ids)
alignment_df = pd.DataFrame([{
    "position": i, "token_id": tid, "token": tok, "attention_mask": mask[i]
} for i,(tid,tok) in enumerate(zip(input_ids,tokens))])
display(alignment_df)
print("Decoded:", processor.tokenizer.decode(input_ids, skip_special_tokens=False))


In [ ]:
# CELL 11 — Verify logits/hidden sequence compatibility
print("Logits shape:", tuple(outputs.logits.shape))
print("Hidden shape:", tuple(last_hidden.shape))
if outputs.logits.shape[:2] != last_hidden.shape[:2]:
    raise RuntimeError("Logits and contextual hidden states do not share [batch, sequence] dimensions.")
print("PASS: one teacher-forced forward exposes aligned per-position logits and contextual states.")


## 5. Cross-attention probe

Cross-attention is recorded because it was central to the Minor attention analysis. It is a diagnostic only; it is **not** the semantic target representation for the Major consistency loss. Any 24×24 visualization is a later analysis resize, not the native tensor shape.


In [ ]:
# CELL 12 — Inspect native cross-attention
cross_attentions = getattr(outputs, "cross_attentions", None)
tensor_attn = None
if cross_attentions is None:
    print("No cross_attentions field returned by this API/checkpoint.")
else:
    print("Cross-attention layers:", len(cross_attentions))
    for i,a in enumerate(cross_attentions):
        print(f"Layer {i}:", tuple(a.shape) if torch.is_tensor(a) else type(a).__name__)
    tensor_attn = next((a for a in reversed(cross_attentions) if torch.is_tensor(a)), None)
    if tensor_attn is not None and tensor_attn.ndim == 4:
        b,h,t,v = tensor_attn.shape
        print("Last cross-attention shape:", tuple(tensor_attn.shape))
        print("Visual-token count:", v)


## 6. Identify LoRA candidates from the actual model

Do not guess target names such as `q_proj` or `query`. Export the actual linear attention-related module names from this checkpoint. The training configuration should be locked only after this report is inspected.


In [ ]:
# CELL 13 — Candidate attention/projection modules
candidates=[]
for name,module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        lname=name.lower()
        if any(k in lname for k in ["query","key","value","q_proj","k_proj","v_proj","out_proj","attention","crossattention"]):
            candidates.append({"name":name,"type":type(module).__name__,"in_features":module.in_features,"out_features":module.out_features})
candidate_df=pd.DataFrame(candidates)
print("Candidate linear modules:", len(candidate_df))
display(candidate_df.head(100))
candidate_path=DIAGNOSTICS_DIR/"blip_lora_candidate_modules.csv"
candidate_df.to_csv(candidate_path,index=False)
print("Saved:",candidate_path)


## 7. Freeze the pretrained BLIP backbone

The Major uses a frozen base backbone plus LoRA adapters and a trainable token projection. This notebook verifies the frozen-backbone state; adapter insertion belongs to the training implementation.


In [ ]:
# CELL 14 — Parameter inventory and freeze
total_params=sum(p.numel() for p in model.parameters())
for p in model.parameters(): p.requires_grad=False
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total BLIP parameters:", f"{total_params:,}")
print("Trainable after freeze:", f"{trainable:,}")
assert trainable==0


## 8. Frozen BLIP generation

This wrapper replaces the Minor multi-model loop. It has no CDN/random-image fallback and does not turn failures into valid captions.


In [ ]:
# CELL 15 — Generation function
def generate_caption(image):
    model.eval()
    batch=processor(images=image, return_tensors="pt")
    batch={k:v.to(DEVICE) if torch.is_tensor(v) else v for k,v in batch.items()}
    with torch.no_grad():
        ids=model.generate(**batch, **GENERATION_CONFIG)
    caption=processor.tokenizer.decode(ids[0], skip_special_tokens=True).strip()
    if not caption: raise RuntimeError("BLIP generated an empty caption.")
    return caption

print("Probe generation:", generate_caption(probe_image))


In [ ]:
# CELL 16 — Generate standardized frozen-BLIP predictions
rows=[]
for _,row in tqdm(df.iterrows(), total=len(df), desc="Frozen BLIP generation"):
    image_path=Path(row["image_path"])
    image=Image.open(image_path).convert("RGB")
    caption=generate_caption(image)
    rows.append({
        "image_id":row["image_id"],
        "image_path":str(image_path),
        "condition":"frozen_blip",
        "generated_caption":caption,
        "reference_captions":json.dumps(get_reference_captions(row), ensure_ascii=False),
    })
predictions_df=pd.DataFrame(rows)
prediction_path=PREDICTIONS_DIR/"frozen_blip_predictions.csv"
predictions_df.to_csv(prediction_path,index=False)
print("Saved:",prediction_path)
display(predictions_df.head())


## 9. Save the architecture report

This report is the handoff to the Major training implementation. It prevents silent assumptions about hidden dimension, visual-token count, cross-attention shape, and LoRA candidates.


In [ ]:
# CELL 17 — Architecture report
report={
    "checkpoint":BLIP_CHECKPOINT,
    "model_class":type(model).__name__,
    "processor_class":type(processor).__name__,
    "device":str(DEVICE),
    "decoder_hidden_size_config":hidden_size,
    "decoder_hidden_size_runtime":int(D),
    "teacher_forced_hidden_shape":list(last_hidden.shape),
    "logits_shape":list(outputs.logits.shape),
    "vision_hidden_size":vision_hidden_size,
    "vision_image_size":image_size,
    "vision_patch_size":patch_size,
    "cross_attention_layers":len(cross_attentions) if cross_attentions is not None else None,
    "cross_attention_last_shape":list(tensor_attn.shape) if tensor_attn is not None else None,
    "lora_candidate_module_count":len(candidates),
    "generation_config":GENERATION_CONFIG,
}
report_path=DIAGNOSTICS_DIR/"blip_architecture_report.json"
with open(report_path,"w",encoding="utf-8") as f: json.dump(report,f,indent=2)
print("Saved:",report_path)
print(json.dumps(report,indent=2))


## 10. Final validation checklist

Notebook 02 is complete only when all checks pass.

- Exact Major BLIP checkpoint loads.
- Real images load without fallback.
- Teacher-forced forward pass succeeds.
- Contextual decoder hidden states are available.
- Runtime hidden dimension is recorded.
- Logits and hidden states have compatible sequence dimensions.
- Native cross-attention shape is recorded when available.
- Actual LoRA candidate modules are exported.
- Frozen BLIP generation works.
- Predictions use the standardized schema.
- Architecture report is saved.

**Never** use simulated images, CDN fallbacks, `[ERROR]` predictions, multi-model generation, static word embeddings, cross-attention as the consistency target, or a hard-coded native `24×24` visual grid.


In [ ]:
# CELL 18 — Machine-readable validation
checks={
    "checkpoint_loaded":model is not None,
    "teacher_forced_hidden_available":last_hidden is not None,
    "hidden_is_3d":last_hidden.ndim==3,
    "logits_hidden_sequence_match":outputs.logits.shape[:2]==last_hidden.shape[:2],
    "candidate_modules_exported":candidate_path.exists(),
    "architecture_report_exported":report_path.exists(),
    "predictions_exported":prediction_path.exists(),
}
for name,ok in checks.items(): print(("PASS" if ok else "FAIL"),"—",name)
if not all(checks.values()): raise RuntimeError("Notebook 02 validation failed.")
print("\nNOTEBOOK 02 COMPLETE — architecture verified and frozen-BLIP predictions produced.")
